In [1]:
import numpy as np
import trimesh as tm
from matplotlib import pyplot as plt
from trimesh.creation import icosphere
from scipy import interpolate
%matplotlib inline

In [2]:
s = icosphere(6, 2.)
def c2p(x):
    theta = np.arccos(x[2]/(np.sqrt(x[0]**2 + x[1]**2 + x[2]**2)))
    phi = np.arctan2(x[0],x[1])
    if phi < 0.:
        phi += 2*np.pi
        
    return 180*phi/np.pi,180*theta/np.pi

def c2p_arr(x):
    theta = np.arccos(x[:,2]/(np.sqrt(x[:,0]**2 + x[:,1]**2 + x[:,2]**2)))
    phi = np.arctan2(x[:,0],x[:,1])
    phi[np.argwhere(phi<0.)] += 2*np.pi
        
    return 180*phi/np.pi,180*theta/np.pi

class FarFieldPattern(object):
    def __init__(self, pattern_theta_file, pattern_phi_file):
        d_theta = np.load(pattern_theta_file)
        d_phi = np.load(pattern_phi_file)
        print(d_theta.shape)
        self.pattern = np.zeros((d_theta.shape[0], d_theta.shape[1], d_theta.shape[2], 3), dtype=complex) #freq, phi, theta
        print(self.pattern.shape)
        self.pattern[:,:,:,1] = d_theta
        self.pattern[:,:,:,2] = d_phi
        
        
class IsotropicPattern(object):
    def __init__(self, theta, phi):
        
        self.pattern = np.zeros((361, 181, 501, 3), dtype=complex)
        print(self.pattern.shape)
        self.pattern[:,:,:,1] = theta
        self.pattern[:,:,:,2] = phi
        

In [3]:
p_theta = "/home/garten/Projekte/Programmierung/rimfax/2021-02-23_RIMFAX_AOIH_0_0.66_2.0_e_theta_filtered_f_close.npy"
p_phi = "/home/garten/Projekte/Programmierung/rimfax/2021-02-23_RIMFAX_AOIH_0_0.66_2.0_e_phi_filtered_f_close.npy"

d_theta = np.load(p_theta)
d_theta[0,0,0]

(-6.540412524724867-1.9798641570780564j)

In [22]:
pattern = FarFieldPattern(p_theta, p_phi)

(361, 181, 501)
(361, 181, 501, 3)


In [5]:
pattern = IsotropicPattern(1,0)

(361, 181, 501, 3)


In [6]:
pattern.pattern.shape

(361, 181, 501, 3)

In [7]:
from tqdm import tqdm
import itertools
from scipy.interpolate import RegularGridInterpolator

def get_interpolation_points(vertices, freq):
    p,t  = c2p_arr(vertices)

    print(t.shape)
    print(p.shape)
    temp = np.array(list(itertools.product(np.array(list(zip(p,t))), freq)), dtype=object)
    points = np.array(np.column_stack([np.array(temp[:,0].tolist(), dtype=float), temp[:,1]]), dtype=float)

    print(points.shape)
    return points

def save_interpolated_pattern(pattern, points, theta, phi, freq, fname):
   
    
    data_interpolator = RegularGridInterpolator((phi, theta, freq), pattern.pattern)

    new_pattern_data = data_interpolator(points)    
    #print(new_pattern_data)
    np.save(f'{fname}_ffpattern.npy', new_pattern_data)
    return new_pattern_data
        

In [23]:
origin_sphere_resolution = 6
origin_sphere = icosphere(origin_sphere_resolution, 2.)

theta = np.linspace(0,180, 181)
phi = np.linspace(0,360,361)
freq = np.linspace(150e6,1200e6, 501) 

In [24]:
#vertices = get_close_vertices(origin_sphere)

In [25]:
points = get_interpolation_points(origin_sphere.vertices, freq)

(40962,)
(40962,)
(20521962, 3)


In [26]:
complete_pattern = save_interpolated_pattern(pattern, points, theta, phi, freq, "test_rimfax_res6")